In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_parquet("../5DATA/dataset/TRAIN_stage1")

---
# 다중공선성 check

In [3]:
X = df.drop(columns=["fraud","id"])
corr = X.corr().abs()

In [4]:
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={0: "corr"})
    .query("corr >= 0.8")
    .sort_values("corr", ascending=False)
)

high_corr_pairs


,level_0,level_1,corr
45,amount_vs_client_avg_diff,amount_deviation,0.899707
186,card_fraud_last1,card_fraud_last3,0.889763
185,card_fraud_last1,client_fraud_last1,0.849746
1,log_abs_amount,amount_vs_client_avg_diff,0.802383


| level_0                   | level_1                   | corr     | keep               | drop                      | rationale                                |
| ------------------------- | ------------------------- | -------- | ------------------ | ------------------------- | ---------------------------------------- |
| amount_vs_client_avg_diff | amount_deviation          | 0.899707 | amount_deviation   | amount_vs_client_avg_diff | deviation이 정규화·스케일 보정 관점에서 더 일반화된 이상치 지표 |
| card_fraud_last1          | card_fraud_last3          | 0.889763 | card_fraud_last3   | card_fraud_last1          | 3회 누적 이력이 재발 패턴을 더 안정적으로 반영              |
| card_fraud_last1          | client_fraud_last1        | 0.849746 | client_fraud_last1 | card_fraud_last1          | 고객 단위 패턴이 카드 단위보다 상위 개념이며 정보 범위가 더 넓음    |
| log_abs_amount            | amount_vs_client_avg_diff | 0.802383 | log_abs_amount     | amount_vs_client_avg_diff | 절대 금액 분포는 기본 스케일 정보로 유지 가치 높음            |


In [5]:
df.drop(columns=["amount_vs_client_avg_diff", "card_fraud_last1"], inplace=True)

---
# SHAP

In [6]:
LABEL_COL = "fraud"  

y = df[LABEL_COL].astype(int)
X = df.drop(columns=[LABEL_COL])

# 1) 컬럼 타입 자동 추정

cat_cols = [c for c in X.columns if str(X[c].dtype) in ("object", "category")]
num_cols = [c for c in X.columns if c not in cat_cols]

print("num_cols:", len(num_cols), "cat_cols:", len(cat_cols))


# 2) Train/Valid split

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3) 간단 전처리 파이프

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)), 
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
    ],
    remainder="drop",
)


num_cols: 23 cat_cols: 0


In [7]:
X_train.columns

Index(['id', 'log_abs_amount', 'high_amount', 'amount_deviation', 'has_error',
       'err_bad_cvv', 'err_bad_card_number', 'err_bad_expiration',
       'card_error_last1', 'client_error_last1', 'client_fraud_last1',
       'card_fraud_last3', 'tx_hour', 'tx_month', 'hour_cos',
       'is_highrisk_weekday', 'seconds_since_prev_tx',
       'card_velocity_spike_ratio', 'card_mcc_is_new', 'client_mcc_is_new',
       'card_merchant_is_new', 'client_merchant_is_new',
       'merchant_is_new_x_has_error'],
      dtype='object')

In [8]:
X_train.drop("id", axis=1, inplace=True)
X_valid.drop("id", axis=1, inplace=True)

In [9]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

X_train_lgb = X_train.copy()
X_valid_lgb = X_valid.copy()

dtrain = lgb.Dataset(X_train_lgb, label=y_train, free_raw_data=False)
dvalid = lgb.Dataset(X_valid_lgb, label=y_valid, free_raw_data=False)

params = dict(
    objective="binary",
    metric=["auc", "average_precision"],
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=200,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    verbosity=-1,
)

bst = lgb.train(
    params,
    dtrain,
    num_boost_round=5000,
    valid_sets=[dvalid],
    valid_names=["valid"],
    callbacks=[
        lgb.early_stopping(200, verbose=True),
        lgb.log_evaluation(0),
    ],
)

print("best_iter:", bst.best_iteration)
print("best_score:", bst.best_score)

pred_valid = bst.predict(X_valid_lgb, num_iteration=bst.best_iteration)
print("LGB AUC:", roc_auc_score(y_valid, pred_valid))
print("LGB PR-AUC:", average_precision_score(y_valid, pred_valid))


# SHAP with tqdm (batch version)

from tqdm.auto import tqdm

sv = X_valid_lgb

batch_size = 2000
all_contrib = []

print("\nComputing SHAP (LightGBM native)...")

for i in tqdm(range(0, len(sv), batch_size)):
    batch = sv.iloc[i:i+batch_size]
    contrib = bst.predict(batch, pred_contrib=True)
    all_contrib.append(contrib[:, :-1])  # 마지막 열은 bias

shap_values = np.vstack(all_contrib)

imp = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=sv.columns
).sort_values(ascending=False)

print("\nTop SHAP features:\n", imp.head(30))


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[171]	valid's auc: 0.995236	valid's average_precision: 0.902843
best_iter: 171
best_score: defaultdict(<class 'collections.OrderedDict'>, {'valid': OrderedDict([('auc', np.float64(0.9952357434156509)), ('average_precision', np.float64(0.9028434966095871))])})
LGB AUC: 0.9952357434156509
LGB PR-AUC: 0.902843496609586

Computing SHAP (LightGBM native)...


  0%|          | 0/61 [00:00<?, ?it/s]


Top SHAP features:
 log_abs_amount                 0.534315
tx_hour                        0.391605
card_merchant_is_new           0.383930
client_merchant_is_new         0.237402
amount_deviation               0.188302
card_fraud_last3               0.159181
seconds_since_prev_tx          0.139092
card_velocity_spike_ratio      0.103295
tx_month                       0.086514
client_mcc_is_new              0.082282
hour_cos                       0.080337
client_fraud_last1             0.079686
is_highrisk_weekday            0.065933
card_mcc_is_new                0.031997
err_bad_cvv                    0.006800
high_amount                    0.002730
card_error_last1               0.001824
has_error                      0.001772
client_error_last1             0.001353
merchant_is_new_x_has_error    0.001084
err_bad_expiration             0.000942
err_bad_card_number            0.000865
dtype: float64


---
# Attention 

In [10]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

feature_names = list(X_train.columns)
n_features = len(feature_names)

scaler = StandardScaler()
Xtr = scaler.fit_transform(X_train.values.astype(np.float32))
Xva = scaler.transform(X_valid.values.astype(np.float32))

ytr = y_train.values.astype(np.float32)
yva = y_valid.values.astype(np.float32)

class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

train_loader = DataLoader(TabDataset(Xtr, ytr), batch_size=4096, shuffle=True, num_workers=0)
valid_loader = DataLoader(TabDataset(Xva, yva), batch_size=8192, shuffle=False, num_workers=0)

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1) Attention layer (weights 반환)

class AttnEncoderLayer(nn.Module):
    def __init__(self, d_model=64, n_heads=4, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, return_attn=False):
        # x: [B, T, D]
        attn_out, attn_w = self.mha(x, x, x, need_weights=True, average_attn_weights=False)
        x = self.ln1(x + attn_out)
        x = self.ln2(x + self.ff(x))
        if return_attn:
            # attn_w: [B, heads, T, T]
            return x, attn_w
        return x


# 2) Tabular Transformer (CLS 토큰 사용)

class TabularAttentionModel(nn.Module):
    def __init__(self, n_features, d_model=64, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # feature별 1->d 투영 (각 feature마다 별도의 linear)
        self.feat_proj = nn.ModuleList([nn.Linear(1, d_model) for _ in range(n_features)])

        # CLS token
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls, std=0.02)

        self.layers = nn.ModuleList([AttnEncoderLayer(d_model, n_heads, dropout) for _ in range(n_layers)])

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, X, return_attn=False):
        # X: [B, F]
        B, F = X.shape

        # feature tokens 만들기: [B, F, D]
        toks = []
        for j in range(F):
            xj = X[:, j:j+1]                 # [B, 1]
            toks.append(self.feat_proj[j](xj))  # [B, D]
        tok = torch.stack(toks, dim=1)       # [B, F, D]

        # CLS 붙이기: [B, 1+F, D]
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, tok], dim=1)

        attn_all = []
        for layer in self.layers:
            if return_attn:
                x, attn = layer(x, return_attn=True)
                attn_all.append(attn)
            else:
                x = layer(x, return_attn=False)

        # CLS representation으로 예측
        cls_repr = x[:, 0, :]               # [B, D]
        logit = self.head(cls_repr).squeeze(1)

        if return_attn:
            return logit, attn_all  # list of [B, heads, T, T]
        return logit


# 3) 학습 루프

model = TabularAttentionModel(n_features=n_features, d_model=64, n_heads=4, n_layers=2, dropout=0.1).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
loss_fn = nn.BCEWithLogitsLoss()

def eval_model():
    model.eval()
    ps, ys = [], []
    with torch.no_grad():
        for xb, yb in valid_loader:
            xb = xb.to(device)
            logit = model(xb)
            prob = torch.sigmoid(logit).cpu().numpy()
            ps.append(prob)
            ys.append(yb.numpy())
    p = np.concatenate(ps)
    t = np.concatenate(ys)
    return roc_auc_score(t, p), average_precision_score(t, p)

EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    model.train()
    pbar = tqdm(train_loader, desc=f"train epoch {epoch}", leave=False)
    for xb, yb in pbar:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        logit = model(xb)
        loss = loss_fn(logit, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        pbar.set_postfix(loss=float(loss.detach().cpu()))

    auc, pr = eval_model()
    print(f"[epoch {epoch}] valid AUC={auc:.5f}  PR-AUC={pr:.5f}")


# 4) Attention 추출 → 컬럼 중요도
# - CLS(0번 토큰)에서 각 feature 토큰으로 가는 attention을 사용

def extract_feature_attention_importance(model, loader, n_batches=50):
    model.eval()
    # 누적: feature별 attention 합
    att_sum = np.zeros((n_features,), dtype=np.float64)
    cnt = 0

    with torch.no_grad():
        for b, (xb, yb) in enumerate(tqdm(loader, desc="extract attention", total=min(n_batches, len(loader)))):
            if b >= n_batches:
                break
            xb = xb.to(device)

            logit, attn_all = model(xb, return_attn=True)
            attn = attn_all[-1]  # [B, heads, T, T]

            # CLS -> feature 토큰 attention: query=0, key=1..F
            # shape: [B, heads, F]
            cls_to_feat = attn[:, :, 0, 1:]  

            # heads 평균, batch 평균 → [F]
            score = cls_to_feat.mean(dim=1).mean(dim=0).cpu().numpy()
            att_sum += score
            cnt += 1

    att_mean = att_sum / max(cnt, 1)
    imp = pd.Series(att_mean, index=feature_names).sort_values(ascending=False)
    return imp

att_imp = extract_feature_attention_importance(model, valid_loader, n_batches=50)
print("\nTop Attention features:\n", att_imp.head(30))


train epoch 1:   0%|          | 0/120 [00:00<?, ?it/s]

[epoch 1] valid AUC=0.98903  PR-AUC=0.83930


train epoch 2:   0%|          | 0/120 [00:00<?, ?it/s]

[epoch 2] valid AUC=0.99090  PR-AUC=0.86619


train epoch 3:   0%|          | 0/120 [00:00<?, ?it/s]

...

[epoch 4] valid AUC=0.99244  PR-AUC=0.88202


train epoch 5:   0%|          | 0/120 [00:00<?, ?it/s]

[epoch 5] valid AUC=0.99297  PR-AUC=0.88942


extract attention:   0%|          | 0/15 [00:00<?, ?it/s]


Top Attention features:
 amount_deviation               0.112755
card_merchant_is_new           0.082258
tx_hour                        0.080306
tx_month                       0.079477
seconds_since_prev_tx          0.061055
client_fraud_last1             0.056700
card_fraud_last3               0.055317
err_bad_expiration             0.047074
client_mcc_is_new              0.043743
high_amount                    0.042072
hour_cos                       0.039570
log_abs_amount                 0.038550
card_velocity_spike_ratio      0.035375
is_highrisk_weekday            0.032162
err_bad_card_number            0.027747
client_error_last1             0.026329
card_error_last1               0.024968
has_error                      0.022542
client_merchant_is_new         0.018237
card_mcc_is_new                0.016739
merchant_is_new_x_has_error    0.012466
err_bad_cvv                    0.011603
dtype: float64


In [11]:
compare = pd.DataFrame({
    "shap_mean_abs": imp,        
    "attn_cls2feat": att_imp
}).fillna(0.0)

compare["shap_rank"] = compare["shap_mean_abs"].rank(ascending=False, method="min")
compare["attn_rank"] = compare["attn_cls2feat"].rank(ascending=False, method="min")
compare["rank_gap"] = compare["attn_rank"] - compare["shap_rank"]

print(compare.sort_values("shap_mean_abs", ascending=False).head(40))


                             shap_mean_abs  attn_cls2feat  shap_rank  \
log_abs_amount                    0.534315       0.038550        1.0   
tx_hour                           0.391605       0.080306        2.0   
card_merchant_is_new              0.383930       0.082258        3.0   
client_merchant_is_new            0.237402       0.018237        4.0   
amount_deviation                  0.188302       0.112755        5.0   
card_fraud_last3                  0.159181       0.055317        6.0   
seconds_since_prev_tx             0.139092       0.061055        7.0   
card_velocity_spike_ratio         0.103295       0.035375        8.0   
tx_month                          0.086514       0.079477        9.0   
client_mcc_is_new                 0.082282       0.043743       10.0   
hour_cos                          0.080337       0.039570       11.0   
client_fraud_last1                0.079686       0.056700       12.0   
is_highrisk_weekday               0.065933       0.032162       

---
# Ablation

In [12]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

params = dict(
    objective="binary",
    metric="auc",
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=200,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    verbosity=-1,
)

def train_eval(X_tr, y_tr, X_va, y_va):
    dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
    dvalid = lgb.Dataset(X_va, label=y_va, free_raw_data=False)

    bst = lgb.train(
        params,
        dtrain,
        num_boost_round=3000,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )

    pred = bst.predict(X_va, num_iteration=bst.best_iteration)

    return {
        "auc": roc_auc_score(y_va, pred),
        "prauc": average_precision_score(y_va, pred),
        "best_iter": bst.best_iteration,
    }


In [13]:
base_result = train_eval(X_train, y_train, X_valid, y_valid)
print("BASE:", base_result)

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[232]	valid_0's auc: 0.995397
BASE: {'auc': 0.9953965646415402, 'prauc': 0.9024508815705057, 'best_iter': 232}


In [14]:
drop_one_results = []

for col in tqdm(X_train.columns):
    cols = [c for c in X_train.columns if c != col]

    res = train_eval(
        X_train[cols],
        y_train,
        X_valid[cols],
        y_valid,
    )

    drop_one_results.append({
        "dropped_feature": col,
        "auc_drop": base_result["auc"] - res["auc"],
        "prauc_drop": base_result["prauc"] - res["prauc"],
        "auc": res["auc"],
        "prauc": res["prauc"],
    })

drop_one_df = pd.DataFrame(drop_one_results)\
    .sort_values("auc_drop", ascending=False)

drop_one_df

  0%|          | 0/22 [00:00<?, ?it/s]

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[235]	valid_0's auc: 0.995361
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[76]	valid_0's auc: 0.995353
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[48]	valid_0's auc: 0.994847
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[93]	valid_0's auc: 0.99533
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[93]	valid_0's auc: 0.995504
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[79]	valid_0's auc: 0.995402
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[79]	valid_0's auc: 0.995402
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[78]	valid_0's auc: 0

,dropped_feature,auc_drop,prauc_drop,auc,prauc
10,card_fraud_last3,0.004013,0.034819,0.991384,0.867632
11,tx_hour,0.000787,0.009288,0.994610,0.893163
2,amount_deviation,0.000549,0.003890,0.994847,0.898561
20,client_merchant_is_new,0.000520,0.001241,0.994876,0.901210
15,seconds_since_prev_tx,0.000297,0.012259,0.995100,0.890191
18,client_mcc_is_new,0.000279,0.004396,0.995118,0.898055
21,merchant_is_new_x_has_error,0.000252,0.000693,0.995145,0.901757
14,is_highrisk_weekday,0.000229,0.002048,0.995168,0.900403
9,client_fraud_last1,0.000172,0.006744,0.995225,0.895707
8,client_error_last1,0.000117,0.000456,0.995280,0.901995


### KEEP

| feature                   | 근거                                                            |
| ------------------------- | ------------------------------------------------------------- |
| log_abs_amount            | SHAP 1위. 금액 스케일 자체가 사기 확률의 기본 분포를 형성하는 핵심 변수                  |
| tx_hour                   | SHAP·Attention 모두 상위권, ablation 시 PRAUC 유의미 감소. 시간대 패턴 설명력 높음 |
| amount_deviation          | Attention 1위 + ablation 시 성능 감소. 개인 기준 대비 이상치 신호로 독립적 가치 존재   |
| card_fraud_last3          | ablation 시 AUC·PRAUC 감소폭 최대. 재발 이력은 가장 강력한 구조적 신호             |
| seconds_since_prev_tx     | PRAUC 감소폭 큼. 단기 거래 간격 기반 velocity 패턴 유지 필요                    |
| client_merchant_is_new    | SHAP 상위 + 제거 시 성능 감소. 고객-가맹점 신규성은 맥락 정보 제공                    |
| client_mcc_is_new         | SHAP 중상위 + PRAUC 감소. 고객 소비 카테고리 변화 신호 보존 가치 있음                |
| client_fraud_last1        | Attention 상위 + PRAUC 감소. 단기 고객 리스크 보조 신호                      |
| card_velocity_spike_ratio | SHAP 상위권 + 제거 시 성능 감소. 카드 단위 급격한 활동 변화 반영                     |
| is_highrisk_weekday       | 제거 시 PRAUC 감소. 요일 기반 위험 패턴 보조 설명 변수                           |
| hour_cos                  | 시간 주기성 보존 목적. sin/cos 구조 중 하나는 유지해 시간 순환성 표현 필요               |

### DROP

| feature                     | 근거                                                       |
| --------------------------- | -------------------------------------------------------- |
| merchant_is_new_x_has_error | SHAP·Attention 모두 하위권, 제거 시 성능 영향 미미                     |
| client_error_last1          | 중요도 낮고 ablation 영향 거의 없음                                 |
| card_error_last1            | 중요도 낮고 PRAUC 변화 미미                                       |
| has_error                   | 단독 설명력 낮고 파생 변수와 중복                                      |
| high_amount                 | log_abs_amount·amount_deviation과 정보 중복, 제거 영향 거의 없음      |
| card_mcc_is_new             | 제거 시 성능 소폭 개선 경향. client_mcc_is_new와 정보 중복               |
| tx_month                    | 제거 시 성능 영향 거의 없음. 시간 패턴은 hour 계열이 설명                     |
| err_bad_expiration          | SHAP 하위권, 제거해도 성능 저하 없음                                  |
| err_bad_card_number         | 중요도 낮고 제거 영향 없음                                          |
| card_merchant_is_new        | SHAP 상위였으나 제거 시 성능 개선. client_merchant_is_new와 강한 중복 가능성 |
| err_bad_cvv                 | 제거 시 AUC 개선. 노이즈 변수 가능성 높음                               |


---